# Первая практика — мини-анализ HMDA

## Подготовка

In [1]:
from pathlib import Path
import sys
import pandas as pd

print("Python:", sys.executable)
print("pandas:", pd.__version__)
print("Рабочая папка:", Path.cwd())
DATA = Path("../data")
assert DATA.is_dir(), "Рядом с папкой ноутбука должна находиться data/"

Python: C:\Users\Windows 11\Desktop\Проектики\Основы Анализа Данных\.venv\Scripts\python.exe
pandas: 2.2.2
Рабочая папка: C:\Users\Windows 11\Desktop\Проектики\Основы Анализа Данных\lessons\01-introduction\work


## Маленький пример reading_example.csv

In [2]:
example = pd.read_csv(
    DATA / "reading_example.csv",
    dtype={"lei": "string", "county_code": "string"},
)
example

,source_row,activity_year,lei,county_code,action_taken,loan_amount,income
0,2,2023,549300VZVN841I2ILS84,11001,1,275000.0,72.0
1,3,2023,549300P1347BFCBHJV60,11001,1,585000.0,174.0
2,278,2023,KB1H1DSPRFMYMCUFXT09,11001,2,475000.0,108.0
3,304,2023,549300VZVN841I2ILS84,11001,2,905000.0,353.0
4,21,2023,549300RPOGWJRH63HS39,11001,3,595000.0,184.0
5,27,2023,JJKC32MCHWDI71265Z06,11001,3,1805000.0,667.0


## A. Разделитель CSV

In [10]:
wrong_sep = pd.read_csv(
    DATA / "reading_example.csv",
    sep =";",
    dtype = {"lei": "string", "county_code": "string"},    
)

print("wrong_sep:", wrong_sep.shape)
print("example:", example.shape)

wrong_sep: (6, 1)
example: (6, 7)


## B. Частичное чтение

In [11]:
preview = pd.read_csv(
    DATA / "reading_example.csv",
    nrows=3,
    dtype={"lei": "string", "county_code": "string"},
)

small_columns = pd.read_csv(
    DATA / "reading_example.csv",
    usecols=["action_taken", "income"],
    dtype={"lei": "string", "county_code": "string"},
)

preview.shape, small_columns.shape

((3, 7), (6, 2))

## Выбор Series и DataFrame

In [14]:
outcomes = example["action_taken"]
print(type(outcomes))
outcomes

<class 'pandas.core.series.Series'>


0    1
1    1
2    2
3    2
4    3
5    3
Name: action_taken, dtype: int64

## Выбор нескольких столбцов

In [15]:
names = ["source_row", "action_taken", "income"]
selected = example[names]
print(type(selected))
selected.head(6)

<class 'pandas.core.frame.DataFrame'>


,source_row,action_taken,income
0,2,1,72.0
1,3,1,174.0
2,278,2,108.0
3,304,2,353.0
4,21,3,184.0
5,27,3,667.0


## C. Выбор столбцов и просмотр

In [16]:
amount_view = example[["source_row", "loan_amount"]].head(2)

amount_view

,source_row,loan_amount
0,2,275000.0
1,3,585000.0


In [17]:
print("Размер:", example.shape)
print("Названия столбцов:", example.columns)
example.info()

Размер: (6, 7)
Названия столбцов: Index(['source_row', 'activity_year', 'lei', 'county_code', 'action_taken',
       'loan_amount', 'income'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   source_row     6 non-null      int64  
 1   activity_year  6 non-null      int64  
 2   lei            6 non-null      string 
 3   county_code    6 non-null      string 
 4   action_taken   6 non-null      int64  
 5   loan_amount    6 non-null      float64
 6   income         6 non-null      float64
dtypes: float64(2), int64(3), string(2)
memory usage: 468.0 bytes


## Пример расчёта одобрений

In [ ]:
approved_flag = outcomes.isin([1, 2])
approved_flag

## Количество и доля одобрений

In [ ]:
n_approved = approved_flag.sum()
n_decisions = len(example)
assert n_decisions > 0
small_approval_rate = n_approved / n_decisions
print(n_approved, n_decisions)
print(f"Одобрено: {100 * small_approval_rate:.2f}%")

## D. Доля выдач на маленьком примере

In [18]:
issued_flag = example["action_taken"].eq(1)

n_issued = issued_flag.sum()
small_origination_rate = n_issued / len(example)

print("Признаки выдачи:", issued_flag.tolist())
print("Количество выдач:", n_issued)
print("Всего записей:", len(example))
print("Доля выдач:", small_origination_rate)
print("Доля выдач, %:", small_origination_rate * 100)

Признаки выдачи: [True, True, False, False, False, False]
Количество выдач: 2
Всего записей: 6
Доля выдач: 0.3333333333333333
Доля выдач, %: 33.33333333333333


## Основной набор decisions.csv

In [20]:
decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
full_outcomes = decisions["action_taken"]
full_n = len(decisions)
print(decisions.shape)
full_outcomes.value_counts(dropna=False)

(6830, 7)


action_taken
1    5956
3     691
2     183
Name: count, dtype: int64

## E. Две доли на основном наборе

In [21]:
full_approved = full_outcomes.isin([1, 2]).sum()
full_issued = full_outcomes.eq(1).sum()

full_approval_rate = full_approved / full_n
full_origination_rate = full_issued / full_n

print("Всего записей:", full_n)
print("Одобрено записей:", full_approved)
print("Доля одобрений, %:", full_approval_rate * 100)
print("Выдано кредитов:", full_issued)
print("Доля выдач, %:", full_origination_rate * 100)

Всего записей: 6830
Одобрено записей: 6139
Доля одобрений, %: 89.88286969253294
Выдано кредитов: 5956
Доля выдач, %: 87.203513909224


## Проверка входных данных

In [22]:
assert full_n > 0
assert full_outcomes.isna().sum() == 0
assert full_outcomes.isin([1, 2, 3]).sum() == full_n
income_missing = decisions["income"].isna().sum()
print("Пропущен доход:", income_missing)

Пропущен доход: 342


## F. Баланс и диапазоны

In [23]:
full_denied = full_outcomes.eq(3).sum()

assert full_approved + full_denied == full_n
assert 0 <= full_issued <= full_approved <= full_n
assert 0 <= full_approval_rate <= 1
assert 0 <= full_origination_rate <= 1

print("Все проверки пройдены.")

Все проверки пройдены.


## G. Расчёт из отчёта

In [24]:
approval_rate = full_outcomes.eq(1).sum() / full_n
approval_rate

0.8720351390922401

## G. Исправление условия

In [27]:
approval_rate_corrected = full_outcomes.isin([1, 2]).sum() / full_n

approval_rate

0.8720351390922401

## G. Исправление названия

In [28]:
origination_rate_corrected = full_outcomes.eq(1).sum() / full_n

origination_rate_corrected

0.8720351390922401